# Retrieval Engine

This engine is responsible for semantic search.

Inputs

- Chunk Objects
- Embedding Objects
- User Question

Output

- Retrieved Chunk Objects

The engine does not generate answers.

Its only responsibility is finding the most relevant knowledge.

# Step 1 - Environment

Mount Google Drive and configure project paths.

Every engine begins with the same environment setup.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import json
from pathlib import Path

import numpy as np

from sentence_transformers import SentenceTransformer
from sentence_transformers import util

In [31]:
ROOT = Path("/content/drive/MyDrive/MicroBrain")

DATA = ROOT / "Data"

RAW = DATA / "raw"

PROCESSED = DATA / "processed"

CHUNKS = DATA / "chunks"

EMBEDDINGS = DATA / "embeddings"

RETRIEVAL = DATA / "retrieval"

VECTORDB = DATA / "vectordb"

METADATA = DATA / "metadata"

for folder in [
    RAW,
    PROCESSED,
    CHUNKS,
    EMBEDDINGS,
    RETRIEVAL,
    VECTORDB,
    METADATA
]:
    folder.mkdir(parents=True, exist_ok=True)

In [4]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(
    MODEL_NAME
)

print("Environment Ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Environment Ready


# Step 2 - Validate Project Structure

Verify all required folders exist before continuing.

In [5]:
print(CHUNKS.exists())

print(EMBEDDINGS.exists())

print(METADATA.exists())

True
True
True


# Step 3 - Discover Available Documents

Load the chunk registry.

The registry tells the engine which chunk datasets are available.

In [6]:
registry_path = METADATA / "chunks.json"

with open(registry_path, "r", encoding="utf-8") as file:

    registry = json.load(file)

print(type(registry))

<class 'list'>


## Step X – Load Chunk Objects

The retrieval engine does not search the metadata.

Instead, the metadata tells us where the actual chunk files are.

This step loads every chunk object into memory so they can later be matched against embedding vectors.

In [9]:
chunk_objects = []

for item in registry:

    chunk_path = CHUNKS / item["chunk_file"]

    with open(chunk_path, "r", encoding="utf-8") as file:

        chunk_objects.extend(
            json.load(file)
        )

print(type(chunk_objects))
print(len(chunk_objects))

<class 'list'>
25


## Step – Load Embedding Registry

The retrieval engine loads the embedding registry to discover every embedding file produced by the Embedding Engine.

The registry itself contains only metadata.

The actual vectors are stored separately inside the embedding files.

In [10]:
embedding_registry_path = METADATA / "embeddings.json"

with open(embedding_registry_path, "r", encoding="utf-8") as file:
    embedding_registry = json.load(file)

print(type(embedding_registry))
print(len(embedding_registry))

<class 'dict'>
1


## Step – Load Embedding Objects

Each registry entry points to one embedding object file.

This step loads every embedding object into memory.

Each embedding object contains

- embedding_id
- document_id
- chunk_id
- model
- dimensions
- vector

In [13]:
embedding_objects = []

for item in embedding_registry["documents"]:

    embedding_path = EMBEDDINGS / item["embedding_file"]

    with open(embedding_path, "r", encoding="utf-8") as file:

        embedding_objects.extend(
            json.load(file)
        )

print(type(embedding_objects))
print(len(embedding_objects))


<class 'list'>
25


## Step 7 – Build Search Index

Each embedding belongs to one chunk.

Instead of searching two separate lists, combine them into a single searchable structure.

Each search record contains:

- chunk_id
- document_id
- text
- embedding vector

This becomes the Retrieval Engine's in-memory search index.

In [14]:
search_index = []

chunk_lookup = {}

for chunk in chunk_objects:

    chunk_lookup[chunk["chunk_id"]] = chunk

for embedding in embedding_objects:

    chunk = chunk_lookup[embedding["chunk_id"]]

    search_index.append({

        "document_id": embedding["document_id"],

        "chunk_id": embedding["chunk_id"],

        "text": chunk["content"]["text"],

        "vector": embedding["vector"]

    })

print(type(search_index))
print(len(search_index))

<class 'list'>
25


## Step 8 – Encode the User Query

To compare a user's question with stored document chunks, the question must be converted into an embedding using the **same embedding model** used during indexing.

This ensures that both the query and document chunks exist in the same vector space and can be compared using cosine similarity.

Input

User Question

Output

Query Embedding (384-dimensional vector)

In [15]:
question = "How do transformers convert words into vectors?"

print(question)

How do transformers convert words into vectors?


In [16]:
query_embedding = embedding_model.encode(question)

print(type(query_embedding))
print()

print(query_embedding.shape)

<class 'numpy.ndarray'>

(384,)


## Step 9 – Compute Similarity Scores

Compare the query embedding against every stored document embedding.

Each comparison produces a cosine similarity score between:

-1 → opposite meaning

0 → unrelated

1 → identical meaning

Higher scores indicate more relevant chunks.

In [17]:
from sentence_transformers import util


In [18]:
results = []

for item in search_index:

    similarity = util.cos_sim(
        query_embedding,
        item["vector"]
    ).item()

    results.append({

        "score": similarity,

        "chunk_id": item["chunk_id"],

        "document_id": item["document_id"],

        "text": item["text"]

    })

print(type(results))
print(len(results))

<class 'list'>
25


## Step 10 – Rank Search Results

After computing similarity scores, rank every chunk from most relevant to least relevant.

The highest cosine similarity indicates the chunk whose meaning is closest to the user's question.

The Retrieval Engine will return only the highest-ranked chunks.

In [19]:
results = sorted(
    results,
    key=lambda item: item["score"],
    reverse=True
)

print(type(results))
print(len(results))

<class 'list'>
25


## Step 11 – Retrieve the Most Relevant Chunks

Instead of passing the entire document to the language model, retrieve only the highest-ranking chunks.

These chunks form the context that will later be injected into the prompt.

This is the core idea behind Retrieval-Augmented Generation (RAG).

In [20]:
TOP_K = 3

retrieved_chunks = results[:TOP_K]

print(type(retrieved_chunks))
print(len(retrieved_chunks))

<class 'list'>
3


## Step 12 – Build Context

The retrieved chunks are combined into a single context block.

This context will later be passed to the Prompting Engine, which constructs the final prompt for the language model.


In [22]:
context = "\n\n".join(

    chunk["text"]

    for chunk in retrieved_chunks

)

print(type(context))
print()

print(len(context))

<class 'str'>

1504


## Step 13 – Save Retrieved Context

The Retrieval Engine saves the retrieved context as a reusable object.

Future engines should never recompute retrieval.

Instead they simply load the retrieval output and continue the pipeline.

In [23]:
import uuid
import json

retrieval_object = {

    "retrieval_id": str(uuid.uuid4()),

    "question": question,

    "top_k": TOP_K,

    "retrieved_chunks": retrieved_chunks,

    "context": context

}

## Step 14 – Save Retrieval Object

Store the retrieval object inside the Outputs directory.

This object becomes the direct input for the Prompting Engine.

In [26]:
import uuid

retrieval_object = {

    "retrieval_id": str(uuid.uuid4()),

    "question": question,

    "top_k": TOP_K,

    "retrieved_chunks": retrieved_chunks,

    "context": context

}

print(retrieval_object.keys())

dict_keys(['retrieval_id', 'question', 'top_k', 'retrieved_chunks', 'context'])


In [27]:
retrieval_file = RETRIEVAL / f"{retrieval_object['retrieval_id']}.json"

with open(
    retrieval_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        retrieval_object,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Saved")

print(retrieval_file)

Saved
/content/drive/MyDrive/MicroBrain/Data/retrieval/e0348682-db79-4bc2-a545-b6eb73723a1a.json


Step 15 - Update Retrieval Registry

In [34]:
registry_path = METADATA / "retrievals.json"

if registry_path.exists():

    with open(
        registry_path,
        "r",
        encoding="utf-8"
    ) as file:

        registry = json.load(file)

else:

    registry = []

In [35]:
registry.append({

    "retrieval_id": retrieval_object["retrieval_id"],

    "question": question,

    "top_k": TOP_K,

    "retrieval_file": retrieval_file.name

})

In [36]:
with open(
    registry_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        registry,
        file,
        indent=4
    )

print(registry_path)

/content/drive/MyDrive/MicroBrain/Data/metadata/retrievals.json


## Step 17 – Validate Retrieval Registry

Verify that the retrieval registry correctly references every retrieval object created by this engine.

In [37]:
registry_path = METADATA / "retrievals.json"

with open(registry_path, "r", encoding="utf-8") as file:
    registry = json.load(file)

print(type(registry))
print(len(registry))

print()

print(registry[0])

<class 'list'>
1

{'retrieval_id': 'e0348682-db79-4bc2-a545-b6eb73723a1a', 'question': 'How do transformers convert words into vectors?', 'top_k': 3, 'retrieval_file': 'e0348682-db79-4bc2-a545-b6eb73723a1a.json'}


## Step 18 – Final Validation

Verify that all retrieval artifacts have been created successfully.

The Retrieval Engine should produce:

• Retrieval Object

• Retrieval Registry

The Prompting Engine will consume these outputs directly.

In [38]:
print("Retrieval Objects")
print()

for file in RETRIEVAL.glob("*.json"):
    print(file.name)

print()

print("Metadata")
print()

for file in METADATA.glob("*retrieval*"):
    print(file.name)

Retrieval Objects

e0348682-db79-4bc2-a545-b6eb73723a1a.json

Metadata

retrievals.json
